# 2 — Query the Graph

What three layers buy you that one does not.

```
question ─▶ registry facts ─▶ traverse ─▶ chunk ids ─▶ Pinecone ─▶ answer
                                                        (text)
```

| § | Stage |
|---|---|
| 0 | Setup |
| 1 | Registry queries — no vector equivalent |
| 2 | Provenance — facts against claims |
| 3 | Registry to document, and back |
| 4 | Local retrieval |
| 5 | Global — community summaries |
| 6 | Four question shapes, compared |

---
## 0. Setup

In [ ]:
# MUST run before any `from graph_rag import ...` below, and before the
# os.environ.setdefault(...) block that follows — setdefault only fills a
# gap, so .env needs to be loaded first for its values to take precedence
# over the placeholders below rather than the other way around.
from dotenv import load_dotenv

load_dotenv()


In [ ]:
import os

# Everything else comes from .env via load_dotenv() above. This checks only
# what actually has no usable fallback:
#
#   OPENAI_API_KEY, PINECONE_API_KEY   no default exists — os.environ[...]
#                                       raises a bare KeyError deep inside
#                                       the first API call that needs it
#   NEO4J_PASSWORD                     store.driver() rejects an EMPTY
#                                       password with a clear error — a
#                                       non-empty PLACEHOLDER would defeat
#                                       that check silently and fail later
#                                       as a Neo4j auth error instead, so
#                                       this is checked here directly rather
#                                       than defaulted to anything at all
#
# NEO4J_URI, NEO4J_USER, NEO4J_DATABASE, INDEX_NAME and the rest all have
# working defaults already, in config.py itself (local Docker, "neo4j",
# "rag-docs") — restating the same defaults here would just be a second
# place for them to drift out of agreement with the first. Override any of
# them in .env, not here.
missing = [name for name in ("OPENAI_API_KEY", "PINECONE_API_KEY", "NEO4J_PASSWORD")
          if not os.getenv(name)]
if missing:
    raise RuntimeError(f"missing from .env (or the environment): {', '.join(missing)}")
print("required secrets present")


In [ ]:
import pandas as pd

from graph_rag import answer, chunks as chunk_reader, config, store

index = chunk_reader.index()
neo = store.driver()
session = neo.session(database=config.NEO4J_DATABASE)


def cypher(query: str, **params) -> pd.DataFrame:
    """Run a query and return the rows as a frame."""
    return pd.DataFrame([dict(row) for row in session.run(query, **params)])


stats = store.summary(session)
print(f"{sum(stats['nodes'].values())} nodes, "
      f"{sum(stats['relationships'].values())} relationships")
print(stats["nodes"])

---
## 1. Registry queries

Not slower than a vector search, not approximate — **impossible**. There is no
embedding of "shares a sponsor with".

In [ ]:
# Sponsors running more than one trial in the corpus. A join, not a similarity.
cypher("""
MATCH (s:Sponsor)<-[:SPONSORED_BY]-(t:Trial)
WITH s, collect(t.nctId) AS trials
WHERE size(trials) > 1
RETURN s.name AS sponsor, size(trials) AS n, trials
ORDER BY n DESC
""")

In [ ]:
# Trials connected through a shared MeSH term — the controlled vocabulary doing the
# entity resolution that fuzzy matching would get wrong.
cypher("""
MATCH (a:Trial)-[:INDEXED_AS]->(m:MeSHTerm)<-[:INDEXED_AS]-(b:Trial)
WHERE a.nctId < b.nctId
RETURN m.term AS shared_concept, a.nctId AS trial_a, b.nctId AS trial_b
ORDER BY shared_concept LIMIT 20
""")

In [ ]:
# Geography. Registry data, exact, and unavailable from any amount of chunk
# retrieval because no single passage lists it this way.
cypher("""
MATCH (t:Trial)-[:CONDUCTED_IN]->(c:Country)
RETURN c.name AS country, count(t) AS trials, collect(t.nctId)[..5] AS sample
ORDER BY trials DESC LIMIT 15
""")

---
## 2. Provenance — facts against claims

Every node knows where it came from. That is the difference between a graph you
would deploy and a demo.

In [ ]:
cypher("""
MATCH (n) WHERE n.source IS NOT NULL
RETURN n.source AS source, labels(n)[0] AS label, count(*) AS n
ORDER BY source, n DESC
""")

In [ ]:
# A query that accepts only facts. Nothing a model inferred can reach this answer.
cypher("""
MATCH (t:Trial)-[:TESTS]->(d:Drug)
WHERE t.source = 'registry' AND d.source = 'registry'
RETURN t.nctId AS trial, t.phase AS phase, collect(d.name) AS drugs
ORDER BY trial LIMIT 15
""")

---
## 3. Registry to document, and back

The `ABOUT` edge joins the two. A traversal can begin at a registry fact and end at
the passage of the protocol that discusses it — which is the whole reason both
layers exist in one graph.

In [ ]:
# From a sponsor, to its trials, to the documents, to the sections that discuss
# safety. Four hops across three layers, in one query.
cypher("""
MATCH (s:Sponsor)<-[:SPONSORED_BY]-(t:Trial)<-[:ABOUT]-(d:Document)
MATCH (d)-[:HAS_SECTION]->(sec:Section)
WHERE toLower(sec.heading) CONTAINS 'safety'
   OR toLower(sec.heading) CONTAINS 'adverse'
RETURN s.name AS sponsor, t.nctId AS trial, sec.heading AS section
LIMIT 20
""")

In [ ]:
# And down to the chunks, which is where the text is. The graph names them; the
# vector store returns what they say.
rows = cypher("""
MATCH (t:Trial)<-[:ABOUT]-(:Document)-[:HAS_SECTION]->(s:Section)-[:HAS_CHUNK]->(c:Chunk)
WHERE toLower(s.heading) CONTAINS 'eligibility'
   OR toLower(s.heading) CONTAINS 'inclusion'
RETURN t.nctId AS trial, s.heading AS section, c.chunkId AS chunk_id, c.page AS page
LIMIT 6
""")

if not rows.empty:
    texts = chunk_reader.fetch_text(index, list(rows["chunk_id"]))
    for _, row in rows.iterrows():
        meta = texts.get(row["chunk_id"], {})
        print(f"{row['trial']}  p{row['page']}  {row['section'][:40]}")
        print(f"  {meta.get('text', '')[:200]}\n")
else:
    print("no eligibility sections found — check section headings in the graph")

---
## 4. Local retrieval

Match entities in the question, traverse, collect the chunks that mention them,
fetch the text.

In [ ]:
result = answer.local(session, index, "What assessments happen at week 12?")

print("terms matched:", result["terms"])
if result["reason"]:
    print("no answer:", result["reason"])
else:
    print(f"{len(result['chunks'])} chunks reached\n")
    print(result["answer"])

In [ ]:
# The failure worth seeing. A question naming no entity in the graph retrieves
# nothing, where dense retrieval would still return its best guess.
miss = answer.local(session, index, "What is the general outlook?")
print("terms:", miss["terms"])
print("reason:", miss["reason"] or "answered")

**Graph retrieval is exact and brittle; vector retrieval is fuzzy and always
answers.** Neither is better — they fail differently, which is why production
systems run both and merge the results.

---
## 5. Global — community summaries

Cluster the entities, summarise each cluster once, answer corpus-level questions
from the summaries.

Top-k retrieval cannot answer *"what are the themes across all of this"* — it
samples five chunks from three thousand and calls that the corpus. A capability gap,
not an efficiency gap.

In [ ]:
summaries = answer.build_summaries(session, min_size=3, top=8)

for community in summaries[:3]:
    print(f"=== {community['type']} cluster around {community['seed']} "
          f"({community['size']} entities) ===")
    print(community["summary"], "\n")

In [ ]:
result = answer.glob(summaries, "What are the main themes across this corpus?")
print(f"answered from {result.get('clusters')} clusters\n")
print(result["answer"])

---
## 6. Four question shapes

Each is answered well by a different mode. Running them together is what shows which
to reach for, and where each fails.

In [ ]:
# "Which sponsors appear in more than one trial?" is NOT a good local() test —
# it is an aggregate question ("which ones appear MORE THAN ONCE"), and local()
# seeds a traversal from ONE named entity. No amount of fixing what counts as a
# seed changes that; an aggregate belongs in raw Cypher, like cell 5 above. The
# fair local() test is a question that NAMES a registry entity directly — which
# is exactly what searching only extracted-layer .key properties used to break.
QUESTIONS = [
    ("What are the inclusion criteria for this trial?",  "single passage"),
    ("What does Ludwig Institute for Cancer Research sponsor?", "registry entity"),
    ("What assessments happen at week 12?",              "extracted relationship"),
    ("What are the main themes across this corpus?",     "corpus-level"),
]

rows = []
for question, shape in QUESTIONS:
    local_result = answer.local(session, index, question)
    rows.append({
        "question": question[:46],
        "shape": shape,
        "terms": len(local_result["terms"]),
        "chunks": len(local_result["chunks"]),
        "graph": "no entity matched" if local_result["reason"] else "answered",
    })

pd.DataFrame(rows)


### What to expect

**Single passage** — vector wins. The graph adds noise by pulling in every chunk
that mentions a common entity.

**Registry join** — graph only, and exactly right. No model involved, so no
possibility of a wrong answer.

**Extracted relationship** — graph, but trust it only as far as §6 of notebook 1
said you should. This is the layer with an accuracy score attached.

**Corpus-level** — community summaries only.

### Then measure it

Everything here is mechanism. Add graph-shaped questions to a labelled set, tag each
by shape, and report recall per shape against vector retrieval alone.

The registry layer needs no such caution — it is correct by construction. The
extracted layer does, and you already have its number.

In [ ]:
session.close()
neo.close()